# Exploratory analysis: dynamic microdynamics

This notebook is outside the main paper workflow. Existing results and metric variants are historical/exploratory, not the authoritative paper results. Original analysis cells and outputs are retained. Some legacy sections require selective execution; this is not a verified clean-run pipeline.

The main workflow is in `notebooks/paper/`. This notebook may write legacy exports under `analysis_exports/`; the paper pipeline uses its own `outputs/paper/` directory.

Plot defaults now come from `plot_style.py`. Prior static image outputs were cleared; rerun plot cells after their prerequisites to see the shared style. Specialized heatmap scales and animations retain their own semantic encodings.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data/wired").is_dir() and (p / "paper").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from this repository or a notebook directory inside it.")
os.chdir(PROJECT_ROOT)  # Preserve project-relative paths when launched from a subdirectory.
print("Project root:", PROJECT_ROOT)

# Shared project plotting conventions.
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from plot_style import (
    apply_style, expertise_palette, EXPERTISE_COLORS, LEVEL_LABELS,
    METRIC_LABELS, MODEL_COLORS, MODEL_MARKERS, PRIMARY, NEUTRAL,
    plot_metric_trajectories, save_figure,
)
apply_style()


# Dynamic conversation microdynamics

This notebook contains only the pipeline used to connect local conversational organization to global semantic geometry:

1. load and embed utterances;
2. collapse consecutive utterances into speaker turns;
3. calculate continuous sustained directional innovation (SDI);
4. measure variation around directions that are actually sustained;
5. model bias-corrected participation ratio (PR) using controls, expertise, and microdynamics.

Run from top to bottom. The microdynamic measures are calculated continuously and do not require topic or episode segmentation.

## 1. Setup and configuration

In [1]:
# Run once if these packages are not already available.
%pip install -q sentence-transformers dimensionality statsmodels scipy seaborn


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.stats import chi2
from sentence_transformers import SentenceTransformer
from dimensionality import participation_ratio
import statsmodels.formula.api as smf

apply_style()
pd.set_option("display.max_columns", 100)

EPSILON = 1e-12
PROJECT_ROOT = Path.cwd()  # change if the notebook is not run from the project root
WIRED_DIR = PROJECT_ROOT / "data" / "wired"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "microdynamics"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

LEVEL_ORDER = ["child", "teenager", "undergraduate", "graduate", "expert"]
LEVEL_MAP = {
    12: (0, "child"), 13: (1, "teenager"), 14: (2, "undergraduate"),
    15: (3, "graduate"), 16: (4, "expert"),
}

# SDI
SDI_HISTORY_WINDOW = 4
SDI_SUBSPACE_DIM = 2
SDI_PERSISTENCE_HORIZON = 2
SUSTAINED_VARIATION_HORIZON = 4
LEARN_CONTEXT_WEIGHTS = True

REQUIRE_FULLY_MATCHED_VIDEOS = True
MIN_TURNS_FOR_PIPELINE = (
    SDI_HISTORY_WINDOW
    + max(SDI_PERSISTENCE_HORIZON, SUSTAINED_VARIATION_HORIZON)
    + 1
)

print("Data directory:", WIRED_DIR)
print("Minimum turns per conversation:", MIN_TURNS_FOR_PIPELINE)

/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data directory: /Users/yume/PycharmProjects/horizons_2/data/wired
Minimum turns per conversation: 9


## 2. Load and standardize the transcripts

In [3]:
def parse_wired_path(path):
    video_id = re.sub(r"^wired_", "", path.parent.name)
    match = re.search(r"_(\d+)$", path.stem)
    return {
        "dataset": "wired",
        "video_id": video_id,
        "file_number": int(match.group(1)) if match else None,
        "conversation_id": path.stem,
        "source_file": str(path.relative_to(PROJECT_ROOT)),
    }


csv_paths = sorted(WIRED_DIR.glob("wired_*/*.csv"))
if not csv_paths:
    raise FileNotFoundError(
        f"No transcript CSVs found under {WIRED_DIR}. Set PROJECT_ROOT so that "
        "PROJECT_ROOT/data/wired contains the wired_* folders."
    )

raw_frames = []
for path in csv_paths:
    frame = pd.read_csv(path)
    frame["raw_row_id"] = np.arange(len(frame))
    for column, value in parse_wired_path(path).items():
        frame[column] = value
    raw_frames.append(frame)

wired_raw = pd.concat(raw_frames, ignore_index=True, sort=False)
wired_raw = wired_raw.rename(columns={"Speaker": "speaker_raw", "Utterance": "text"})
wired_raw["text"] = (
    wired_raw["text"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
)
wired_raw = wired_raw[wired_raw["text"].notna() & wired_raw["text"].ne("")].copy()
wired_raw["level"] = wired_raw["file_number"].map(lambda x: LEVEL_MAP[x][0])
wired_raw["level_label"] = wired_raw["file_number"].map(lambda x: LEVEL_MAP[x][1])
wired_raw["level_label"] = pd.Categorical(
    wired_raw["level_label"], categories=LEVEL_ORDER, ordered=True
)

# Speaker 1 is the focal expert; the other participant is the partner.
is_expert = wired_raw["speaker_raw"].str.contains(r"\bSpeaker\s*1\b", case=False, na=False)
wired_raw["speaker"] = np.where(is_expert, "A", "B")
wired_raw["speaker_role"] = np.where(is_expert, "expert", "partner")

display(
    wired_raw.groupby("level_label", observed=True)
    .agg(n_conversations=("conversation_id", "nunique"), n_utterances=("text", "size"))
    .reset_index()
)

,level_label,n_conversations,n_utterances
0,child,21,1132
1,teenager,21,1222
2,undergraduate,21,1337
3,graduate,21,1177
4,expert,21,1513


## 3. Canonical utterances and embeddings

In [4]:
def count_words(text):
    return len(re.findall(r"\b[\w'-]+\b", str(text)))


def create_utterances_table(raw):
    out = raw.copy().reset_index(drop=False).rename(columns={"index": "_original_row"})
    out = out.sort_values(
        ["dataset", "conversation_id", "raw_row_id", "_original_row"], kind="mergesort"
    ).reset_index(drop=True)
    keys = ["dataset", "conversation_id"]
    changed = out.groupby(keys, sort=False)["speaker"].transform(lambda x: x.ne(x.shift()))
    out["turn_id"] = changed.astype(int).groupby([out[k] for k in keys]).cumsum()
    out["utterance_in_turn"] = out.groupby([*keys, "turn_id"], sort=False).cumcount() + 1
    out["utterance_number"] = out.groupby(keys, sort=False).cumcount() + 1
    out["utterance_id"] = (
        out["dataset"].astype(str) + "::" + out["conversation_id"].astype(str)
        + "::utterance_" + out["utterance_number"].astype(str).str.zfill(3)
    )
    out["n_words"] = out["text"].map(count_words).astype(int)
    out["embedding_idx"] = np.arange(len(out), dtype=int)
    return out


utterances = create_utterances_table(wired_raw)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
E_utterances = embedding_model.encode(
    utterances["text"].astype(str).tolist(),
    batch_size=32, show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float32)

assert E_utterances.shape[0] == len(utterances)
assert np.isfinite(E_utterances).all()
assert np.allclose(np.linalg.norm(E_utterances, axis=1), 1.0, atol=1e-5)
print("Utterances:", len(utterances), "Embedding matrix:", E_utterances.shape)

Batches: 100%|██████████| 200/200 [00:04<00:00, 45.88it/s]

Utterances: 6381 Embedding matrix: (6381, 384)


## 4. Collapse utterances into speaker turns and select matched conversations

In [5]:
def unit_vector(vector):
    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)
    return vector / norm if norm > EPSILON else vector


turn_records, turn_vectors = [], []
for values, group in utterances.groupby(
    ["dataset", "conversation_id", "turn_id"], observed=True, sort=False
):
    group = group.sort_values("embedding_idx", kind="stable")
    indices = group["embedding_idx"].astype(int).to_numpy()
    turn_vectors.append(unit_vector(E_utterances[indices].mean(axis=0)))
    turn_records.append({
        "dataset": values[0], "conversation_id": values[1], "turn_id": values[2],
        "video_id": group["video_id"].iloc[0], "level": int(group["level"].iloc[0]),
        "level_label": group["level_label"].iloc[0],
        "speaker_role": group["speaker_role"].iloc[0],
        "turn_text": " ".join(group["text"].astype(str)),
        "n_utterances_in_turn": len(group),
    })

turns_all = pd.DataFrame(turn_records)
E_turns_all = np.vstack(turn_vectors).astype(np.float32)
turns_all["source_turn_embedding_idx"] = np.arange(len(turns_all), dtype=int)
turns_all["turn_order"] = turns_all.groupby(
    ["dataset", "conversation_id"], observed=True, sort=False
).cumcount()

conversation_eligibility = (
    turns_all.groupby(
        ["dataset", "video_id", "conversation_id", "level", "level_label"],
        observed=True,
    ).size().rename("n_turns").reset_index()
)
conversation_eligibility["eligible"] = (
    conversation_eligibility["n_turns"] >= MIN_TURNS_FOR_PIPELINE
)
video_eligibility = (
    conversation_eligibility.groupby(["dataset", "video_id"], observed=True)
    .agg(n_levels=("level_label", "nunique"), all_eligible=("eligible", "all"))
    .reset_index()
)
video_mask = video_eligibility["all_eligible"]
if REQUIRE_FULLY_MATCHED_VIDEOS:
    video_mask &= video_eligibility["n_levels"].eq(5)
eligible_videos = video_eligibility.loc[video_mask, ["dataset", "video_id"]]
eligible_conversations = (
    conversation_eligibility[conversation_eligibility["eligible"]]
    [["dataset", "video_id", "conversation_id"]]
    .merge(eligible_videos, on=["dataset", "video_id"], how="inner")
)

turns = turns_all.merge(
    eligible_conversations, on=["dataset", "video_id", "conversation_id"], how="inner"
).sort_values(["dataset", "conversation_id", "turn_order"]).reset_index(drop=True)
E_turns = E_turns_all[turns["source_turn_embedding_idx"].astype(int).to_numpy()]
turns = turns.drop(columns="source_turn_embedding_idx")
turns["turn_embedding_idx"] = np.arange(len(turns), dtype=int)
analysis_utterances = utterances.merge(
    eligible_conversations, on=["dataset", "video_id", "conversation_id"], how="inner"
).sort_values(["dataset", "conversation_id", "embedding_idx"]).reset_index(drop=True)

assert len(E_turns) == len(turns)
conversation_turn_counts = turns.groupby(
    ["level_label", "conversation_id"], observed=True
).size().rename("n_turns").reset_index()
print("Eligible videos:", eligible_videos["video_id"].nunique())
print("Eligible conversations:", turns["conversation_id"].nunique())
display(conversation_turn_counts.groupby("level_label", observed=True)["n_turns"].agg(["count", "mean", "min"]))

Eligible videos: 19
Eligible conversations: 95


,count,mean,min
level_label,,,
child,19,28.526316,12
expert,19,25.263158,11
graduate,19,24.473684,9
teenager,19,25.105263,11
undergraduate,19,24.315789,13


## 5. Learn a continuous multiscale context

The context for every eligible turn combines the previous turn, a rolling centroid, and the cumulative conversation centroid:

$$
c_t=\operatorname{unit}\left(w_1x_{t-1}+w_2\bar{x}_{t-H:t}+w_3\bar{x}_{0:t}\right),
\quad w_j\geq0,\quad\sum_jw_j=1.
$$

One shared set of weights is learned across all conversations. No episode boundaries are used, so every eligible turn is evaluated in the same continuous temporal frame.

In [6]:
def context_components(X, t):
    history = X[t - SDI_HISTORY_WINDOW:t]
    return np.vstack([
        unit_vector(X[t - 1]),
        unit_vector(history.mean(axis=0)),
        unit_vector(X[:t].mean(axis=0)),
    ])


context_training_examples = []
for _, group in turns.groupby(
    ["dataset", "conversation_id"], observed=True, sort=False
):
    group = group.sort_values("turn_order")
    X = E_turns[group["turn_embedding_idx"].astype(int).to_numpy()]
    for t in range(SDI_HISTORY_WINDOW, len(X)):
        context_training_examples.append((context_components(X, t), X[t]))


def context_loss(weights):
    losses = []
    for components, target in context_training_examples:
        context = unit_vector(weights @ components)
        losses.append(1.0 - np.clip(context @ target, -1.0, 1.0))
    return float(np.mean(losses))


if LEARN_CONTEXT_WEIGHTS and context_training_examples:
    optimization = minimize(
        context_loss, x0=np.repeat(1 / 3, 3), method="SLSQP",
        bounds=[(0.0, 1.0)] * 3,
        constraints={"type": "eq", "fun": lambda w: w.sum() - 1.0},
    )
    if optimization.success:
        CONTEXT_WEIGHTS = optimization.x / optimization.x.sum()
    else:
        warnings.warn(f"Weight optimization failed: {optimization.message}; using equal weights.")
        CONTEXT_WEIGHTS = np.repeat(1 / 3, 3)
else:
    CONTEXT_WEIGHTS = np.repeat(1 / 3, 3)

display(pd.DataFrame({
    "component": ["previous_turn", "rolling_centroid", "conversation_cumulative_centroid"],
    "weight": CONTEXT_WEIGHTS,
}))
print("Mean prediction cosine distance:", round(context_loss(CONTEXT_WEIGHTS), 4))

,component,weight
0,previous_turn,0.123321
1,rolling_centroid,0.278896
2,conversation_cumulative_centroid,0.597783


Mean prediction cosine distance: 0.5222


## 6. Continuous SDI and precision around sustained directions

For every eligible turn, innovation is separated into the part already represented by the recent rolling subspace and a new-direction component:

$$
i_t=x_t-c_t,\qquad r_t=(I-U_tU_t^\top)i_t,\qquad D_t=\lVert r_t\rVert.
$$

If $P_t$ measures subsequent continuation along the unit direction $v_t=r_t/D_t$, sustained directional innovation is

$$
\operatorname{SDI}_t=D_tP_t.
$$

Variation around that direction is measured from subsequent turn-to-turn movements:

$$
V_t^{\mathrm{sust}}=
\frac{\sum_h\lVert(I-v_tv_t^\top)\Delta x_{t+h}\rVert^2}
{\sum_h\lVert\Delta x_{t+h}\rVert^2+\epsilon},
\qquad Q_t^{\mathrm{sust}}=1-V_t^{\mathrm{sust}}.
$$

Thus, SDI measures sustained exploration, while $Q_t^{\mathrm{sust}}$ measures how precisely the following trajectory enacts the direction. Directions are weighted continuously by their SDI rather than selected using a hard episode or persistence threshold.

In [7]:
def calculate_continuous_sdi(X, metadata):
    records = []
    maximum_future_horizon = max(
        SDI_PERSISTENCE_HORIZON,
        SUSTAINED_VARIATION_HORIZON,
    )
    last_t_exclusive = len(X) - maximum_future_horizon

    for t in range(SDI_HISTORY_WINDOW, last_t_exclusive):
        history = X[t - SDI_HISTORY_WINDOW:t]
        context = unit_vector(CONTEXT_WEIGHTS @ context_components(X, t))
        innovation = X[t] - context
        innovation_magnitude = np.linalg.norm(innovation)

        centered_history = history - history.mean(axis=0, keepdims=True)
        _, singular_values, Vt = np.linalg.svd(centered_history, full_matrices=False)
        k = min(SDI_SUBSPACE_DIM, int(np.sum(singular_values > 1e-10)))
        basis = Vt[:k].T if k else np.empty((X.shape[1], 0))
        represented = basis @ (basis.T @ innovation) if k else np.zeros_like(innovation)
        new_direction = innovation - represented
        directional_magnitude = np.linalg.norm(new_direction)
        directional_fraction = directional_magnitude / (innovation_magnitude + EPSILON)

        if directional_magnitude > EPSILON:
            direction_unit = new_direction / directional_magnitude

            future_displacements = (
                X[t + 1:t + 1 + SDI_PERSISTENCE_HORIZON] - context
            )
            continuation = (
                future_displacements @ direction_unit
            ) / (directional_magnitude + EPSILON)
            persistence = float(np.clip(continuation, 0.0, 1.0).mean())

            future_steps = np.diff(
                X[t:t + SUSTAINED_VARIATION_HORIZON + 1], axis=0
            )
            total_movement_energy = float(np.square(future_steps).sum())
            parallel_coefficients = future_steps @ direction_unit
            perpendicular_steps = (
                future_steps - np.outer(parallel_coefficients, direction_unit)
            )
            sustained_variation = float(
                np.square(perpendicular_steps).sum()
                / (total_movement_energy + EPSILON)
            )
            sustained_variation = float(np.clip(sustained_variation, 0.0, 1.0))
            sustained_precision = 1.0 - sustained_variation
        else:
            persistence = 0.0
            sustained_variation = np.nan
            sustained_precision = np.nan

        row = metadata.iloc[t]
        records.append({
            "dataset": row["dataset"], "video_id": row["video_id"],
            "conversation_id": row["conversation_id"], "level": row["level"],
            "level_label": row["level_label"], "turn_order": row["turn_order"],
            "speaker_role": row["speaker_role"],
            "innovation_magnitude": innovation_magnitude,
            "directional_innovation_magnitude": directional_magnitude,
            "directional_innovation_fraction": directional_fraction,
            "directional_persistence": persistence,
            "sdi": directional_magnitude * persistence,
            "sustained_direction_variation": sustained_variation,
            "sustained_direction_precision": sustained_precision,
        })
    return records


sdi_records = []
for _, group in turns.groupby(
    ["dataset", "conversation_id"], observed=True, sort=False
):
    group = group.sort_values("turn_order").reset_index(drop=True)
    X = E_turns[group["turn_embedding_idx"].astype(int).to_numpy()]
    sdi_records.extend(calculate_continuous_sdi(X, group))

sdi_events = pd.DataFrame(sdi_records)
if sdi_events.empty:
    raise RuntimeError("No eligible SDI events; reduce the history or future horizons.")


def summarize_continuous_microdynamics(group):
    usable = group[
        group["sustained_direction_precision"].notna()
    ].copy()
    weights = usable["sdi"].to_numpy(dtype=float)
    if weights.sum() > EPSILON:
        sustained_precision = float(np.average(
            usable["sustained_direction_precision"], weights=weights
        ))
    else:
        sustained_precision = np.nan

    return pd.Series({
        "mean_sdi": group["sdi"].mean(),
        "median_sdi": group["sdi"].median(),
        "directional_innovation_mean": group["directional_innovation_magnitude"].mean(),
        "directional_persistence_mean": group["directional_persistence"].mean(),
        "sustained_direction_precision": sustained_precision,
        "sustained_direction_variation": 1.0 - sustained_precision,
        "n_sdi_events": len(group),
        "total_sdi_weight": group["sdi"].sum(),
    })


microdynamic_summary = (
    sdi_events.groupby(
        ["dataset", "video_id", "conversation_id", "level", "level_label"],
        observed=True,
    ).apply(summarize_continuous_microdynamics).reset_index()
)

role_sdi = sdi_events.pivot_table(
    index=["dataset", "video_id", "conversation_id"],
    columns="speaker_role", values="sdi", aggfunc="mean",
).rename(columns=lambda role: f"mean_sdi_{role}").reset_index()
microdynamic_summary = microdynamic_summary.merge(
    role_sdi, on=["dataset", "video_id", "conversation_id"], how="left"
)
display(microdynamic_summary.head())

/var/folders/1z/hv4_qzyn6556dk5hqb4k6tdh0000gn/T/ipykernel_44835/94158613.py:112: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(summarize_continuous_microdynamics).reset_index()


,dataset,video_id,conversation_id,level,level_label,mean_sdi,median_sdi,directional_innovation_mean,directional_persistence_mean,sustained_direction_precision,sustained_direction_variation,n_sdi_events,total_sdi_weight,mean_sdi_expert,mean_sdi_partner
0,wired,astro,wired_astro_12,0,child,0.447869,0.441767,0.983837,0.459626,0.067110,0.932890,15.0,6.718040,0.496745,0.405103
1,wired,astro,wired_astro_13,1,teenager,0.379385,0.307620,0.983266,0.387213,0.090045,0.909955,5.0,1.896927,0.428069,0.306359
2,wired,astro,wired_astro_14,2,undergraduate,0.391444,0.383140,0.872797,0.449149,0.077920,0.922080,9.0,3.522996,0.360287,0.416370
3,wired,astro,wired_astro_15,3,graduate,0.400948,0.409653,0.929885,0.434406,0.073882,0.926118,21.0,8.419912,0.395379,0.406011
4,wired,astro,wired_astro_16,4,expert,0.300960,0.277681,0.827908,0.360999,0.079600,0.920400,6.0,1.805757,0.362733,0.239186


## 7. Descriptive microdynamic checks

In [8]:
display(
    microdynamic_summary.groupby("level_label", observed=True)[
        ["mean_sdi", "sustained_direction_precision", "n_sdi_events"]
    ].agg(["count", "mean", "std"])
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, value, ylabel in [
    (axes[0], "mean_sdi", "Mean continuous SDI"),
    (axes[1], "sustained_direction_precision", "Precision around sustained directions"),
]:
    sns.boxplot(data=microdynamic_summary, x="level_label", y=value, order=LEVEL_ORDER, ax=ax)
    sns.stripplot(
        data=microdynamic_summary, x="level_label", y=value, order=LEVEL_ORDER,
        color="black", alpha=0.45, size=3, ax=ax,
    )
    ax.set(xlabel="Partner expertise level", ylabel=ylabel)
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

mean_sdi                     sustained_direction_precision  \
                 count      mean       std                         count   
level_label                                                                
child               19  0.435694  0.049497                            19   
expert              19  0.373325  0.059492                            19   
graduate            19  0.406118  0.075122                            19   
teenager            19  0.429217  0.053754                            19   
undergraduate       19  0.430689  0.054171                            19   

                                  n_sdi_events                        
                   mean       std        count       mean        std  
level_label                                                           
child          0.091898  0.015028           19  20.526316  11.281097  
expert         0.086587  0.015442           19  17.263158  13.265838  
graduate       0.086996  0.020294           19  16.473684  10.189147  
teenager       0.086024  0.014633           19  17.105263  15.566684  
undergraduate  0.086158  0.018988           19  16.315789   9.568772

## 8. Global outcome and controls

The macro-geometric outcome is row-bias-corrected participation ratio from the same unit-normalized utterance embeddings. Utterance count and the expert's proportion of utterances are controls. The speaking-share control is also treated as a sensitivity decision because it may itself be part of scaffolding.

In [9]:
pr_records = []
group_columns = ["dataset", "video_id", "conversation_id", "level", "level_label"]
for values, group in analysis_utterances.groupby(group_columns, observed=True, sort=False):
    X = E_utterances[group["embedding_idx"].astype(int).to_numpy()].astype(float)
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    result = participation_ratio(X, return_all=True)
    pr_records.append({
        **dict(zip(group_columns, values)),
        "n_utterances": len(group),
        "expert_utterance_proportion": float(group["speaker_role"].eq("expert").mean()),
        "pr_naive": float(result["naive"]),
        "pr_row_corrected": float(result["row"]),
        "pr_both_corrected": float(result["both"]),
    })

global_geometry = pd.DataFrame(pr_records)
analysis_table = global_geometry.merge(
    microdynamic_summary, on=group_columns, how="inner", validate="one_to_one"
)
analysis_table["match_id"] = (
    analysis_table["dataset"].astype(str) + "::" + analysis_table["video_id"].astype(str)
)
display(analysis_table.head())

,dataset,video_id,conversation_id,level,level_label,n_utterances,expert_utterance_proportion,pr_naive,pr_row_corrected,pr_both_corrected,mean_sdi,median_sdi,directional_innovation_mean,directional_persistence_mean,sustained_direction_precision,sustained_direction_variation,n_sdi_events,total_sdi_weight,mean_sdi_expert,mean_sdi_partner,match_id
0,wired,talia,wired_Talia_12,0,child,27,0.740741,16.181812,42.062965,47.500553,0.422875,0.406607,1.052837,0.404535,0.094843,0.905157,6.0,2.537252,0.434621,0.411130,wired::talia
1,wired,talia,wired_Talia_13,1,teenager,60,0.666667,23.029034,37.464428,41.767522,0.463504,0.452400,1.046492,0.443465,0.095827,0.904173,32.0,14.832135,0.515263,0.411745,wired::talia
2,wired,talia,wired_Talia_14,2,undergraduate,44,0.681818,18.618395,33.477591,36.902616,0.493716,0.487828,1.042558,0.494718,0.077373,0.922627,21.0,10.368030,0.537966,0.445040,wired::talia
3,wired,talia,wired_Talia_15,3,graduate,43,0.651163,17.342065,29.631373,32.306859,0.464215,0.407770,0.966283,0.527922,0.081710,0.918290,18.0,8.355861,0.514367,0.414062,wired::talia
4,wired,talia,wired_Talia_16,4,expert,44,0.363636,21.715177,45.394155,51.792217,0.335576,0.357411,0.895506,0.382812,0.094882,0.905118,9.0,3.020184,0.380050,0.299997,wired::talia


## 9. Three nested mixed models

All models use the same complete-case conversations and a random intercept for matched video/topic:

$$
\begin{aligned}
M_0:&\quad G_c \sim \log n_c+p_{\mathrm{expert},c}+(1|\mathrm{video})\\
M_1:&\quad G_c \sim \log n_c+p_{\mathrm{expert},c}+E_c+(1|\mathrm{video})\\
M_2:&\quad G_c \sim \log n_c+p_{\mathrm{expert},c}+E_c+\mathrm{SDI}_c+\mathrm{Precision}_c+\mathrm{SDI}_c\!\times\!\mathrm{Precision}_c+(1|\mathrm{video}).
\end{aligned}
$$

Selection precision is the SDI-weighted precision of movement around sustained directions, so larger values mean cleaner directional enactment. The interaction tests whether sustained exploration is most strongly associated with global dimensionality when it is also locally precise. ML fits are used for comparisons; the final model is refit with REML.

In [10]:
MODEL_COLUMNS = [
    "pr_row_corrected", "n_utterances", "expert_utterance_proportion", "level",
    "mean_sdi", "sustained_direction_precision", "match_id",
]
model_data = analysis_table.dropna(subset=MODEL_COLUMNS).copy()


def zscore(series):
    sd = series.std(ddof=0)
    if not np.isfinite(sd) or sd <= EPSILON:
        raise ValueError(f"Cannot standardize {series.name}: zero or invalid SD.")
    return (series - series.mean()) / sd


model_data["global_geometry_z"] = zscore(model_data["pr_row_corrected"])
model_data["log_n_utterances"] = np.log(model_data["n_utterances"])
model_data["log_n_z"] = zscore(model_data["log_n_utterances"])
model_data["expert_proportion_z"] = zscore(model_data["expert_utterance_proportion"])
model_data["expertise_z"] = zscore(model_data["level"].astype(float))
model_data["sdi_z"] = zscore(model_data["mean_sdi"])
model_data["selection_precision_z"] = zscore(model_data["sustained_direction_precision"])

FORMULAS = {
    "M0_controls": "global_geometry_z ~ log_n_z + expert_proportion_z",
    "M1_expertise": "global_geometry_z ~ log_n_z + expert_proportion_z + expertise_z",
    "M2_microdynamics": (
        "global_geometry_z ~ log_n_z + expert_proportion_z + expertise_z "
        "+ sdi_z * selection_precision_z"
    ),
}


def fit_mixed_model(formula, data, reml=False):
    model = smf.mixedlm(formula, data=data, groups=data["match_id"])
    errors = []
    for method in ["lbfgs", "powell"]:
        try:
            result = model.fit(reml=reml, method=method, maxiter=2000, disp=False)
            if result.converged:
                return result
        except Exception as error:
            errors.append(error)
    if errors:
        raise errors[-1]
    return result


ml_fits = {
    name: fit_mixed_model(formula, model_data, reml=False)
    for name, formula in FORMULAS.items()
}
print("Complete-case conversations:", len(model_data))
print("Matched videos/topics:", model_data["match_id"].nunique())

Complete-case conversations: 95
Matched videos/topics: 19


/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


In [11]:
def mixed_r2(result):
    fixed_prediction = np.asarray(result.model.exog @ result.fe_params)
    fixed_variance = float(np.var(fixed_prediction, ddof=0))
    random_variance = float(np.asarray(result.cov_re)[0, 0])
    residual_variance = float(result.scale)
    total = fixed_variance + random_variance + residual_variance
    return fixed_variance / total, (fixed_variance + random_variance) / total


comparison_records = []
for name, result in ml_fits.items():
    marginal_r2, conditional_r2 = mixed_r2(result)
    comparison_records.append({
        "model": name, "n": int(result.nobs), "log_likelihood": result.llf,
        "AIC": result.aic, "BIC": result.bic,
        "marginal_R2": marginal_r2, "conditional_R2": conditional_r2,
        "expertise_beta": result.params.get("expertise_z", np.nan),
        "expertise_p": result.pvalues.get("expertise_z", np.nan),
    })
model_comparison = pd.DataFrame(comparison_records)


def likelihood_ratio_test(smaller, larger, label):
    statistic = max(0.0, 2 * (larger.llf - smaller.llf))
    df = len(larger.fe_params) - len(smaller.fe_params)
    return {"comparison": label, "LR": statistic, "df": df, "p_value": chi2.sf(statistic, df)}


likelihood_tests = pd.DataFrame([
    likelihood_ratio_test(ml_fits["M0_controls"], ml_fits["M1_expertise"], "M0 vs M1"),
    likelihood_ratio_test(ml_fits["M1_expertise"], ml_fits["M2_microdynamics"], "M1 vs M2"),
])
display(model_comparison)
display(likelihood_tests)
display(ml_fits["M2_microdynamics"].summary())

final_model_reml = fit_mixed_model(FORMULAS["M2_microdynamics"], model_data, reml=True)
display(final_model_reml.summary())

,model,n,log_likelihood,AIC,BIC,marginal_R2,conditional_R2,expertise_beta,expertise_p
0,M0_controls,95,-115.966051,241.932102,254.701486,0.210854,0.480994,NaN,NaN
1,M1_expertise,95,-109.217322,230.434644,245.757905,0.268974,0.570460,0.387294,0.000114
2,M2_microdynamics,95,-104.786742,227.573484,250.558376,0.333649,0.575917,0.336081,0.001116


,comparison,LR,df,p_value
0,M0 vs M1,13.497458,1,0.000239
1,M1 vs M2,8.861161,3,0.031195


<class 'statsmodels.iolib.summary2.Summary'>
"""
                Mixed Linear Model Regression Results
======================================================================
Model:               MixedLM   Dependent Variable:   global_geometry_z
No. Observations:    95        Method:               ML               
No. Groups:          19        Scale:                0.4058           
Min. group size:     5         Log-Likelihood:       -104.7867        
Max. group size:     5         Converged:            Yes              
Mean group size:     5.0                                              
----------------------------------------------------------------------
                            Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------------
Intercept                    0.004    0.131  0.027 0.979 -0.254  0.261
log_n_z                      0.200    0.090  2.221 0.026  0.024  0.376
expert_proportion_z         -0.131    0.105 -1.249 0.212 -0.336  0.074
expertise_z                  0.336    0.103  3.259 0.001  0.134  0.538
sdi_z                       -0.030    0.101 -0.300 0.764 -0.229  0.168
selection_precision_z       -0.245    0.100 -2.452 0.014 -0.441 -0.049
sdi_z:selection_precision_z  0.007    0.051  0.130 0.896 -0.092  0.106
Group Var                    0.232    0.186                           
======================================================================

"""

/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


<class 'statsmodels.iolib.summary2.Summary'>
"""
                Mixed Linear Model Regression Results
======================================================================
Model:               MixedLM   Dependent Variable:   global_geometry_z
No. Observations:    95        Method:               REML             
No. Groups:          19        Scale:                0.4374           
Min. group size:     5         Log-Likelihood:       -115.6818        
Max. group size:     5         Converged:            Yes              
Mean group size:     5.0                                              
----------------------------------------------------------------------
                            Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------------
Intercept                    0.004    0.137  0.026 0.979 -0.264  0.272
log_n_z                      0.200    0.094  2.137 0.033  0.017  0.383
expert_proportion_z         -0.130    0.109 -1.201 0.230 -0.343  0.082
expertise_z                  0.336    0.107  3.143 0.002  0.127  0.546
sdi_z                       -0.030    0.105 -0.289 0.772 -0.237  0.176
selection_precision_z       -0.244    0.104 -2.359 0.018 -0.448 -0.041
sdi_z:selection_precision_z  0.007    0.052  0.129 0.897 -0.096  0.110
Group Var                    0.253    0.200                           
======================================================================

"""

### Scaffolding sensitivity check

In [12]:
# Speaking share can be a confound, but it can also be part of the scaffolding process.
sensitivity_formulas = {
    "M1_without_speaking_share": "global_geometry_z ~ log_n_z + expertise_z",
    "M2_without_speaking_share": (
        "global_geometry_z ~ log_n_z + expertise_z + sdi_z * selection_precision_z"
    ),
}
sensitivity_fits = {
    name: fit_mixed_model(formula, model_data, reml=False)
    for name, formula in sensitivity_formulas.items()
}
sensitivity_coefficients = pd.concat(
    {
        name: pd.DataFrame({"estimate": fit.params, "p_value": fit.pvalues})
        for name, fit in sensitivity_fits.items()
    }, names=["model", "term"],
).reset_index()
display(sensitivity_coefficients)

/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


,model,term,estimate,p_value
0,M1_without_speaking_share,Intercept,-2.192525e-16,1.000000e+00
1,M1_without_speaking_share,log_n_z,1.777876e-01,3.575356e-02
2,M1_without_speaking_share,expertise_z,4.587857e-01,2.805261e-11
3,M1_without_speaking_share,Group Var,7.215979e-01,3.144029e-02
4,M2_without_speaking_share,Intercept,7.509444e-03,9.554869e-01
5,M2_without_speaking_share,log_n_z,1.815884e-01,4.227201e-02
6,M2_without_speaking_share,expertise_z,4.222292e-01,3.877499e-08
7,M2_without_speaking_share,sdi_z,-5.563322e-02,5.772262e-01
8,M2_without_speaking_share,selection_precision_z,-2.405017e-01,1.645341e-02
9,M2_without_speaking_share,sdi_z:selection_precision_z,1.408022e-02,7.798310e-01


## 10. Micro-to-macro visual checks

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.regplot(
    data=model_data, x="sdi_z", y="global_geometry_z",
    scatter_kws={"alpha": 0.65}, line_kws={"color": "black"}, ax=axes[0],
)
axes[0].set(xlabel="Mean continuous SDI (z)", ylabel="Bias-corrected PR (z)")
sns.regplot(
    data=model_data, x="selection_precision_z", y="global_geometry_z",
    scatter_kws={"alpha": 0.65}, line_kws={"color": "black"}, ax=axes[1],
)
axes[1].set(xlabel="Precision around sustained directions (z)", ylabel="Bias-corrected PR (z)")
plt.tight_layout()
plt.show()

coefficient_rows = []
for model_name, fit in ml_fits.items():
    intervals = fit.conf_int()
    for term in [
        "expertise_z", "sdi_z", "selection_precision_z",
        "sdi_z:selection_precision_z",
    ]:
        if term in fit.params:
            coefficient_rows.append({
                "model": model_name, "term": term, "estimate": fit.params[term],
                "lower": intervals.loc[term, 0], "upper": intervals.loc[term, 1],
            })
coefficient_plot_data = pd.DataFrame(coefficient_rows)

fig, ax = plt.subplots(figsize=(7, 4))
for y, row in coefficient_plot_data.reset_index(drop=True).iterrows():
    ax.errorbar(
        row["estimate"], y,
        xerr=[[row["estimate"] - row["lower"]], [row["upper"] - row["estimate"]]],
        fmt="o", capsize=3,
    )
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_yticks(range(len(coefficient_plot_data)))
ax.set_yticklabels(coefficient_plot_data["model"] + ": " + coefficient_plot_data["term"])
ax.set(xlabel="Standardized coefficient", ylabel="")
plt.tight_layout()
plt.show()

## 11. Save analysis-ready outputs

In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "continuous_sdi_events.csv": sdi_events,
    "microdynamic_summary.csv": microdynamic_summary,
    "micro_to_macro_analysis_table.csv": analysis_table,
    "model_comparison.csv": model_comparison,
    "likelihood_ratio_tests.csv": likelihood_tests,
}
for filename, table in outputs.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)
print("Saved outputs to:", OUTPUT_DIR.resolve())
print("Files:", sorted(outputs))

Saved outputs to: /Users/yume/PycharmProjects/horizons_2/outputs/microdynamics
Files: ['continuous_sdi_events.csv', 'likelihood_ratio_tests.csv', 'micro_to_macro_analysis_table.csv', 'microdynamic_summary.csv', 'model_comparison.csv']
